In [ ]:
import os, json, pandas as pd
from unfairness.utils.config_loader import load_hparams
from unfairness.dataset import ToS, make_dataloaders
from unfairness.utils.misc import mask_ids
from unfairness.trainer import train_one_from_splits
from unfairness.reports.plots import plot_losses, plot_f1 

In [ ]:
CSV_PATH = "local_database/ToS_100/ToS_100.csv"
KB_DIR   = "local_database/KB"
DIST_CFG = "configs/distributed_model_config.json"
FOLD_DIR = "cv_test/torch/fold_1"         
MAX_LEN  = 128
BATCH    = 32
N_SUBSET = 500                            
SEED     = 42

os.makedirs(FOLD_DIR, exist_ok=True)
hparams = load_hparams(DIST_CFG)

In [ ]:
df_full = pd.read_csv(CSV_PATH)
if N_SUBSET is not None and N_SUBSET > 0:
    df = df_full.head(N_SUBSET).copy()
else:
    df = df_full.copy()

splits = {
    "train": [1, 2],
    "val":   [3],
    "test":  [4],
}

In [ ]:
import numpy as np

tr_df = mask_ids(df, splits["train"])
va_df = mask_ids(df, splits["val"])
te_df = mask_ids(df, splits["test"])

tr_csv = os.path.join(FOLD_DIR, "train.csv")
va_csv = os.path.join(FOLD_DIR, "val.csv")
te_csv = os.path.join(FOLD_DIR, "test.csv")

tr_df.to_csv(tr_csv, index=False)
va_df.to_csv(va_csv, index=False)
te_df.to_csv(te_csv, index=False)

tr_df.shape, va_df.shape, te_df.shape

In [ ]:
best_ckpt, metrics = train_one_from_splits(
    train_csv=tr_csv,
    val_csv=va_csv,
    test_csv=te_csv,
    hparams=hparams,
    out_dir=FOLD_DIR,
    max_len=MAX_LEN,
    batch_size=BATCH,
)
print("best_ckpt:", best_ckpt)
print("metrics :", metrics)

In [ ]:
plot_losses(FOLD_DIR, smooth=1, show=True, save_path=os.path.join(FOLD_DIR, "loss.png"))
plot_f1(FOLD_DIR,     smooth=1, show=True, save_path=os.path.join(FOLD_DIR, "f1.png"))

In [ ]:
import torch
from sklearn.metrics import classification_report
from unfairness.utils.config_loader import load_model_and_tokenizer
from unfairness.dataset import ToS, make_dataloaders

lit, tok, kb_struct = load_model_and_tokenizer(
    ckpt_path=best_ckpt,
    max_len=MAX_LEN,
    kb_dir=KB_DIR,
    map_location="cuda",  
)

ds_test = ToS(pd.read_csv(te_csv), tok, MAX_LEN)
_, _, te_loader = make_dataloaders(ds_test, None, ds_test, batch_size=BATCH)

all_probs, all_preds, all_gold = [], [], []
lit.eval()
with torch.no_grad():
    for batch in te_loader:
        logits, *_ = lit(batch["input_ids"])
        probs = logits.sigmoid().cpu()
        preds = (probs > 0.5).to(torch.int)
        all_probs.append(probs)
        all_preds.append(preds)
        all_gold.append(batch["labels_multi"].cpu())

import numpy as np
P = torch.cat(all_preds).numpy()
G = torch.cat(all_gold).numpy()

print("=== Multi-label (A,CH,CR,LTD,TER) ===")
print(classification_report(G, P, target_names=["A","CH","CR","LTD","TER"], zero_division=0))

Pg = (P.max(axis=1) > 0).astype(int)
Gg = (G.max(axis=1) > 0).astype(int)
print("=== General (OR derivado) ===")
print(classification_report(Gg, Pg, target_names=["general"], zero_division=0))